# 01 — V3 Kids Ethnic & Wedding Formal (Lehenga) Dataset Builder
**Isolated Workspace:** `/content/drive/MyDrive/FashionFit_PK_V3_Kids_Formal_Extension/` *(Zero edits to FYP-1 or FYP-2!)*

### What This Notebook Builds:
1. **Public Commercial Catalog Collector (Peraan Wedding Formals & Kids/Mini Collections):**
   - Strictly commercial brand catalog imagery (zero private/personal photos of minors).
   - Portrait filter (`H > W`), resolution (`>= 512px`), sharpness gate, SHA-256 + perceptual `dHash` deduplication.
2. **Roboflow Gold Annotation Export (`roboflow_v3_kids_and_formals.zip`):**
   - Exports up to **100 unique outfits** (50 Kids + 50 Wedding Formals/Lehengas, 1 per SKU) for fast Roboflow 4-class annotation.
3. **Zero-Leakage SKU-Grouped Multi-Mode Paired Triples (`paired_train` / `paired_test`):**
   - Uses your **FYP-2 Gold 4-Class SegFormer-B2** (read-only) to generate BOTH:
     - **Standard Kurta/Choli mask** (`mode = upper`)
     - **Novel Dilated A-Line Flared Mask** (`mode = full_flared`) specifically for wide wedding lehengas/peshwas!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q requests pillow numpy pandas opencv-python-headless tqdm transformers torch


In [ ]:
import hashlib
import io
import json
import shutil
import time
import zipfile
import cv2
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

V3_ROOT = Path('/content/drive/MyDrive/FashionFit_PK_V3_Kids_Formal_Extension')
DATA_DIR = V3_ROOT / '01_KIDS_AND_FORMAL_DATA'
RAW_KIDS = DATA_DIR / 'raw_catalog' / 'kids_ethnic'
RAW_FORMAL = DATA_DIR / 'raw_catalog' / 'wedding_formals'
GOLD_V3_DIR = DATA_DIR / 'gold_v3_masks'
PAIRED_TRAIN = DATA_DIR / 'paired_train'
PAIRED_TEST = DATA_DIR / 'paired_test'

for d in [RAW_KIDS, RAW_FORMAL, GOLD_V3_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Read-only reference to FYP-2 Gold 4-Class SegFormer-B2
FYP2_GOLD_SEG = Path('/content/drive/MyDrive/FashionFit_PK_FYP2_Research/02_SEGFORMER_4CLASS_RUNS/segformer_b2_4class_gold_v1/best_checkpoint')
print('FYP-2 Gold 4-Class SegFormer exists:', FYP2_GOLD_SEG.exists())


In [ ]:
# 1. Collect Clean Commercial Catalog Images from Peraan (Shopify public products.json)
# Automatically separates Kids ("mini", "kid", "girl", "junior") vs Adult Wedding Formals (lehenga/peshwas/formal)
HEADERS = {'User-Agent': 'Mozilla/5.0 (Academic Research; FashionFit-PK FYP)'}

SHOPIFY_SOURCES = [
    ('peraan', 'https://peraan.com/collections/wedding-formals/products.json', 'wedding_formals'),
    ('peraan', 'https://peraan.com/products.json', 'auto_detect'),
    ('limelight_girls', 'https://www.limelight.pk/collections/girls/products.json', 'kids_ethnic'),
]

KIDS_KEYWORDS = ['mini', 'kid', 'kids', 'girl', 'girls', 'junior', 'child', 'toddler', 'baby', 'little']
FORMAL_KEYWORDS = ['lehenga', 'lehnga', 'sharara', 'gharara', 'peshwas', 'bridal', 'formal', 'wedding', 'festive', 'choli', 'maxi', 'frock']

def compute_dhash(im_rgb: Image.Image) -> str:
    gray = np.array(im_rgb.convert('L').resize((9, 8), Image.BILINEAR))
    return ''.join('1' if b else '0' for b in (gray[:, 1:] > gray[:, :-1]).flatten())

seen_sha, seen_dhash = set(), set()
records = []
MAX_PER_DOMAIN = 180
domain_counts = {'kids_ethnic': 0, 'wedding_formals': 0}

for brand, base_url, default_domain in SHOPIFY_SOURCES:
    for page in range(1, 10):
        try:
            r = requests.get(base_url, params={'limit': 50, 'page': page}, headers=HEADERS, timeout=20)
            if r.status_code != 200:
                break
            products = r.json().get('products', [])
            if not products:
                break
        except Exception as e:
            print(f'Stop fetching {base_url} p={page}:', e)
            break

        for prod in products:
            title = str(prod.get('title', '')).lower()
            handle = str(prod.get('handle', 'sku')).lower()
            tags = ' '.join(prod.get('tags', [])).lower() if isinstance(prod.get('tags'), list) else str(prod.get('tags', '')).lower()
            meta_txt = f'{title} {handle} {tags}'

            is_kid = (default_domain == 'kids_ethnic') or any(k in meta_txt for k in KIDS_KEYWORDS)
            domain = 'kids_ethnic' if is_kid else ('wedding_formals' if default_domain == 'wedding_formals' or any(k in meta_txt for k in FORMAL_KEYWORDS) else 'wedding_formals')

            if domain_counts[domain] >= MAX_PER_DOMAIN:
                continue

            sku = f'{brand}_{handle[:40]}'
            imgs = prod.get('images', [])[:2]  # Max 2 distinct views per SKU
            for v_idx, img_obj in enumerate(imgs):
                if domain_counts[domain] >= MAX_PER_DOMAIN:
                    break
                src_url = img_obj.get('src', '')
                if not src_url:
                    continue
                try:
                    resp = requests.get(src_url, headers=HEADERS, timeout=15)
                    if resp.status_code != 200:
                        continue
                    raw = resp.content
                    sha = hashlib.sha256(raw).hexdigest()[:16]
                    if sha in seen_sha:
                        continue
                    im = Image.open(io.BytesIO(raw)).convert('RGB')
                    w, h_px = im.size
                    if h_px < 512 or w < 360 or h_px <= w:
                        continue
                    dh = compute_dhash(im)
                    if dh in seen_dhash:
                        continue
                    gray = cv2.cvtColor(np.array(im), cv2.COLOR_RGB2GRAY)
                    sharp = float(cv2.Laplacian(gray, cv2.CV_64F).var())
                    if sharp < 25.0:
                        continue

                    seen_sha.add(sha)
                    seen_dhash.add(dh)
                    out_folder = RAW_KIDS if domain == 'kids_ethnic' else RAW_FORMAL
                    img_id = f'v3_{domain[:3]}_{sha}'
                    out_p = out_folder / f'{img_id}.jpg'
                    im.save(out_p, quality=95)
                    domain_counts[domain] += 1
                    records.append({
                        'image_id': img_id,
                        'domain': domain,
                        'brand': brand,
                        'sku': sku,
                        'title': prod.get('title', ''),
                        'path': str(out_p),
                        'width': w,
                        'height': h_px,
                        'sharpness': round(sharp, 1),
                    })
                except Exception:
                    continue
        time.sleep(0.3)

# Also scan any manually dropped images inside raw_catalog/kids_ethnic or raw_catalog/wedding_formals
for dom, folder in [('kids_ethnic', RAW_KIDS), ('wedding_formals', RAW_FORMAL)]:
    existing_paths = {r['path'] for r in records}
    for p in sorted(folder.glob('*.*')):
        if str(p) in existing_paths or p.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp'}:
            continue
        try:
            im = Image.open(p).convert('RGB')
            w, h_px = im.size
            if h_px < 512 or w < 360 or h_px <= w:
                continue
            sha = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
            records.append({
                'image_id': p.stem,
                'domain': dom,
                'brand': 'manual_curated',
                'sku': p.stem,
                'title': p.stem,
                'path': str(p),
                'width': w,
                'height': h_px,
                'sharpness': 100.0,
            })
        except Exception:
            continue

df = pd.DataFrame(records)
df.to_csv(DATA_DIR / 'v3_kids_formal_manifest.csv', index=False)
print('=== V3 CATALOG COLLECTION SUMMARY ===')
print(df['domain'].value_counts().to_dict() if len(df) else 'No images found — drop catalog photos into raw_catalog/ folders!')

# Export up to 100 unique SKUs (50 Kids + 50 Wedding Formals) for Roboflow 4-class annotation
if len(df) > 0:
    picks = []
    for dom, q in [('kids_ethnic', 50), ('wedding_formals', 50)]:
        sub = df[df['domain'] == dom].drop_duplicates(subset=['sku'])
        if len(sub) > 0:
            picks.append(sub.sample(n=min(q, len(sub)), random_state=42))
    rf_df = pd.concat(picks).reset_index(drop=True)
    zip_out = DATA_DIR / 'roboflow_v3_kids_and_formals.zip'
    with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
        for _, r in rf_df.iterrows():
            zf.write(Path(r['path']), arcname=f"{r['domain']}__{r['image_id']}.jpg")
    rf_df.to_csv(DATA_DIR / 'roboflow_v3_manifest.csv', index=False)
    print(f'Exported {len(rf_df)} unique Kids + Formal SKUs for Roboflow -> {zip_out}')


In [ ]:
# 2. Build Zero-Leakage SKU-Grouped Multi-Mode Paired Triples (Standard Upper + Dilated Flared Lehenga!)
for split_dir in [PAIRED_TRAIN, PAIRED_TEST]:
    if split_dir.exists():
        shutil.rmtree(split_dir)
    for sub in ['person', 'garment', 'mask', 'mask_flared']:
        (split_dir / sub).mkdir(parents=True, exist_ok=True)

seg_src = str(FYP2_GOLD_SEG) if FYP2_GOLD_SEG.exists() else 'mattmdjaga/segformer_b2_clothes'
print('Loading 4-class SegFormer-B2 from:', seg_src)
seg_proc = SegformerImageProcessor.from_pretrained(seg_src)
seg_model = SegformerForSemanticSegmentation.from_pretrained(seg_src).to(device).eval()
is_4class = (seg_model.config.num_labels == 4)

def make_flared_lehenga_mask(kurta_m: np.ndarray, bottom_m: np.ndarray, protect_m: np.ndarray) -> np.ndarray:
    '''Novel Dilated A-Line Flared Mask: expands lower garment mask laterally so wide wedding lehengas
    are NEVER squeezed into tight straight trousers during virtual try-on!'''
    h, w = kurta_m.shape
    combined = ((kurta_m > 0) | (bottom_m > 0)).astype(np.uint8)
    # Dilate lower-body silhouette horizontally to create natural A-line lehenga flare clearance
    ys = np.where(combined > 0)[0]
    if len(ys) == 0:
        return combined
    y_mid = int(ys.min() + 0.45 * (ys.max() - ys.min()))
    lower_only = combined.copy()
    lower_only[:y_mid, :] = 0
    k_w = max(15, int(w * 0.08))
    k_h = max(7, int(h * 0.02))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_w, k_h))
    flared_lower = cv2.dilate(lower_only, kernel, iterations=1)
    out = ((combined > 0) | (flared_lower > 0)).astype(np.uint8)
    out[protect_m > 0] = 0
    return out

@torch.no_grad()
def parse_garment_masks(im: Image.Image):
    w, h = im.size
    enc = seg_proc(images=im, return_tensors='pt').to(device)
    logits = seg_model(**enc).logits
    pred = F.interpolate(logits, size=(h, w), mode='bilinear', align_corners=False).argmax(dim=1)[0].cpu().numpy()
    if is_4class:
        kurta = (pred == 1).astype(np.uint8)
        bottom = (pred == 2).astype(np.uint8)
        dupatta = (pred == 3).astype(np.uint8)
        protect = (pred == 0).astype(np.uint8)
    else:
        kurta = np.isin(pred, [4, 7]).astype(np.uint8)
        bottom = np.isin(pred, [5, 6]).astype(np.uint8)
        dupatta = (pred == 17).astype(np.uint8)
        protect = np.isin(pred, [1, 2, 3, 9, 10, 11, 14, 15]).astype(np.uint8)
    # Protect upper head region strictly
    head_cut = int(h * 0.16)
    protect[:head_cut, :] = 1
    flared = make_flared_lehenga_mask(kurta, bottom, protect)
    full_garment = ((kurta > 0) | (bottom > 0)).astype(np.uint8)
    return kurta, full_garment, flared

if len(df) > 0:
    rng = np.random.default_rng(42)
    test_skus = set()
    for dom in df['domain'].unique():
        skus = np.array(sorted(df[df['domain'] == dom]['sku'].unique()))
        rng.shuffle(skus)
        n_te = max(5, int(0.18 * len(skus)))
        test_skus.update(skus[:n_te])

    meta_rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Generating V3 zero-leakage multi-mode triples'):
        im = Image.open(row['path']).convert('RGB')
        kurta_m, full_m, flared_m = parse_garment_masks(im)
        if full_m.mean() < 0.08 or full_m.mean() > 0.78:
            continue
        is_test = row['sku'] in test_skus
        sdir = PAIRED_TEST if is_test else PAIRED_TRAIN
        stem = row['image_id']

        im.save(sdir / 'person' / f'{stem}.jpg', quality=95)
        # Save full garment cutout on pure white (#FFFFFF) for full-outfit/lehenga + kids try-on
        arr = np.array(im)
        cutout = np.full_like(arr, 255)
        cutout[full_m == 1] = arr[full_m == 1]
        Image.fromarray(cutout).save(sdir / 'garment' / f'{stem}.png')
        Image.fromarray(full_m * 255).save(sdir / 'mask' / f'{stem}.png')
        Image.fromarray(flared_m * 255).save(sdir / 'mask_flared' / f'{stem}.png')
        meta_rows.append({'image_id': stem, 'domain': row['domain'], 'sku': row['sku'], 'split': 'test' if is_test else 'train'})

    mdf = pd.DataFrame(meta_rows)
    mdf.to_csv(DATA_DIR / 'v3_paired_split_index.csv', index=False)
    print('=== ZERO-LEAKAGE V3 PAIRED SPLIT ===')
    print(mdf.groupby(['split', 'domain']).size())
